# LLM Science Length + TF-IDF Ensemble

Ranks choices with row-normalized option length plus a small prompt/option TF-IDF cosine signal.


In [ ]:
import csv
import math
import re
from collections import Counter
from pathlib import Path

OPTIONS = ("A", "B", "C", "D", "E")
TFIDF_WEIGHT = 0.25
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?")
STOPWORDS = {
    "the", "of", "and", "to", "in", "a", "is", "are", "which", "following",
    "statement", "statements", "accurately", "describes", "describe", "what", "how",
    "by", "for", "with", "as", "an", "on", "from", "that", "it", "this", "its",
    "be", "was", "were", "can", "into", "or", "not", "due", "between", "among",
    "about", "refers", "system", "systems", "object", "objects", "law", "theory",
    "mass", "data", "time", "answer", "option",
}


def find_csv(name):
    candidates = sorted(Path("/kaggle/input").glob(f"**/{name}"))
    if not candidates:
        return None
    return candidates[0]


def read_rows(path):
    if path is None:
        return []
    with path.open(newline="", encoding="utf-8") as src:
        return list(csv.DictReader(src))


def tokens(text):
    return [word.lower() for word in WORD_RE.findall(text or "")]


def content_tokens(text):
    return [word for word in tokens(text) if len(word) >= 3 and word not in STOPWORDS]


def word_count(text):
    return len(tokens(text))


def zscores(values):
    mean = sum(values) / len(values)
    var = sum((value - mean) ** 2 for value in values) / len(values)
    scale = math.sqrt(var) or 1.0
    return [(value - mean) / scale for value in values]


def build_idf(rows):
    docs = []
    for row in rows:
        docs.append(set(content_tokens(row["prompt"])))
        for option in OPTIONS:
            docs.append(set(content_tokens(row[option])))
    df = Counter()
    for doc in docs:
        df.update(doc)
    n_docs = len(docs)
    return {term: math.log((n_docs + 1) / (count + 1)) + 1 for term, count in df.items()}


def tfidf_vector(words, idf):
    counts = Counter(words)
    return {word: count * idf.get(word, 1.0) for word, count in counts.items()}


def cosine(a, b):
    numerator = sum(value * b.get(key, 0.0) for key, value in a.items())
    norm_a = math.sqrt(sum(value * value for value in a.values()))
    norm_b = math.sqrt(sum(value * value for value in b.values()))
    if not norm_a or not norm_b:
        return 0.0
    return numerator / (norm_a * norm_b)


def rank_row(row, idf):
    prompt_vec = tfidf_vector(content_tokens(row["prompt"]), idf)
    length_values = [word_count(row[option]) for option in OPTIONS]
    tfidf_values = [
        cosine(prompt_vec, tfidf_vector(content_tokens(row[option]), idf))
        for option in OPTIONS
    ]
    scores = [
        length_z + TFIDF_WEIGHT * tfidf_z
        for length_z, tfidf_z in zip(zscores(length_values), zscores(tfidf_values))
    ]
    return [
        option for _, option in sorted(zip(scores, OPTIONS), key=lambda item: (-item[0], item[1]))
    ]


train_path = find_csv("train.csv")
test_path = find_csv("test.csv")
if test_path is None:
    raise FileNotFoundError("No test.csv found under /kaggle/input")

train_rows = read_rows(train_path)
test_rows = read_rows(test_path)
idf = build_idf(train_rows + test_rows)
print(f"Using train: {train_path}")
print(f"Using test: {test_path}")
print(f"Rows: train={len(train_rows)} test={len(test_rows)}")

output_path = Path("/kaggle/working/submission.csv")
with output_path.open("w", newline="", encoding="utf-8") as dst:
    writer = csv.DictWriter(dst, fieldnames=["id", "prediction"])
    writer.writeheader()
    for row in test_rows:
        writer.writerow({"id": row["id"], "prediction": " ".join(rank_row(row, idf)[:3])})

print(f"Wrote {output_path}")
